# MTPFN beyond d = 1 and h = id: where does it lie between A-only and the oracle?

## Description

*Written 2026-11-04, before the first run. Issue #20, branch `exp/20-mtpfn`. Follows notebooks 13 and 15 (MTPFN on
the AIR prior with d = 1, h = id).*

**Question.** Does MTPFN still realise most of the possible transfer from a dense related task B to a scarce task A
when (i) the output map h between the tasks is not the identity, and (ii) the input space is two-dimensional? Where
does it lie between its two bounds, and where does it break?

**Prior** (`experimental/mtpfn/hd.py`), f_A(x) = h(g(T_θ(x))), x ∈ [0, 1]^d:
- g: AIR's random tanh MLP, extended to d inputs, on a global ECDF scale per d.
- T_θ: a box-preserving flow.
  - d = 1: AIR's, with J = 2.
  - d = 2: v_1 = x_1(1 − x_1) s (θ_1 + ½ θ_2 cos πx_2), v_2 = x_2(1 − x_2) s (θ_3 + ½ θ_4 cos πx_1). The second
    coefficient of each coordinate couples the coordinates. s gives E|T(x) − x| ≈ 0.12 per coordinate.
- h, latent: identity with probability ½; otherwise affine a + b·v with b ~ U[0.3, 1], a ~ U[0, 1 − b] (H1 of the
  AIR spec: B's function rescaled and shifted).
- Data: B has 128 points; A has n_A ∈ U{1..10} (d = 1) or U{1..20} (d = 2); σ_A, σ_B ~ U[0.005, 0.03]; B is always
  related.

**Models** (one per d; 1.84M parameters each, as notebook 13; 100k steps × 64 tasks; AIR's 400-bin bar output):
- **MTPFN**: target A plus source B.
- **A-only (lower bound)**: the same architecture trained on the same prior with A as its only task. It never sees B:
  the clean, separately trained A-only baseline.

**Oracle (upper bound).** The exact posterior predictive given the true g, σ_A and the prior.
- h is integrated out: a in closed form, b on a 2048-point grid, the id/affine mixture exactly.
- θ: a 101² grid at d = 1; adaptive importance sampling at d = 2 (16384 draws per stage, KDE stages with a 10 % prior
  share), with ESS reported.
- The predictive mixes 16384 posterior draws; its MC error is reported.
- It knows g exactly, while the models see 128 noisy points: a bound, not an attainable target.
- Unit tests: the analytic h integral matches brute-force integration; with h = id the oracle reproduces the exact grid
  predictive.

**Evaluation.**
- 256 fixed tasks per d, drawn on the CPU (`eval_tasks`, seed 777), so every machine sees the same tasks.
- n_A ∈ {1, 2, 3, 5, 10} (d = 1) and {1, 2, 3, 5, 10, 20} (d = 2), nested designs.
- NLL per query on the bar scale; strata h = id and h affine.

**What to look for.** The share of the A-only → oracle gap that MTPFN closes, (NLL_A-only − NLL_MTPFN) /
(NLL_A-only − NLL_oracle), by d, n_A and h stratum, and how often MTPFN is worse than A-only. Expectations, written
before any data:
- **E1:** at d = 1, h = id the share is in the range notebook 15 found (84 % → 46 % over n_A = 1 → 10), perhaps
  somewhat lower because the prior is broader.
- **E2:** an affine h costs MTPFN little: at d = 1 the affine stratum's share is within 10 points of the id stratum's
  at every n_A. h is a global two-number map that the [TASK] summary can carry. At n_A = 1, h and the warp are
  confounded, so both bounds move.
- **E3:** at d = 2 the share is lower than at d = 1 for matched n_A and falls faster with n_A. All transfer passes
  through one summary token per layer, and a 2-D B is harder to summarise.

**Breaking point (defined here):** a cell (d, h stratum, n_A) where MTPFN closes less than 50 % of the gap, or is worse
than A-only on more than 20 % of tasks. This is a description, not a GO/NO-GO: the notebook is a diagnosis.

**Runs.**
- One `hd_train` sweep per d with members {MTPFN, A-only} × seeds; one `hd_eval` per (d, seed).
- Seed 0 first; seeds 1 and 2 after.
- Cost: about 2.5 GPU-h per MTPFN and less per A-only, at most 2 jobs at a time on ulysses.

**Limits.**
- Affine h only: no non-monotone or non-linear h.
- B is always related (Z0): no unrelated-source training (the paper's p = 0.2); the robustness question is separate.
- The d = 2 warp family is small (4 coefficients); 128 B points are sparse in 2-D.
- One model size (not the paper's 23 × 512).
- The oracle is IS-based at d = 2 (check its ESS).


## Results

*Filled in once the runs are in.*